# 01 · Extract and cache all features (run once)

Everything expensive happens here, **once**, and is saved under `CACHE_DIR/features/`. `final.ipynb` only reads these files.

| group (column prefix) | what | needs |
|---|---|---|
| transcripts + timestamps | Whisper (reused from `00_baseline` if already cached) | GPU helps |
| `hand_` | length, vocabulary variety, sentence shape, fillers, repetitions, false starts, subordinators | nothing |
| `flu_` | speech rate, pauses (from word timestamps), ASR confidence | nothing |
| `lm_` | GPT-2 perplexity of the transcript | GPT-2 weights |
| `lt_` | LanguageTool grammar-error counts — **optional** | Java + download |
| `llm_` | zero-shot rubric score from a local LLM — **optional** | LLM weights |
| `text_st`, `text_hf` | sentence-encoder and RoBERTa mean-pooled embeddings of the transcript | weights |
| `audio_wavlm` | WavLM mean+std-pooled embedding of the audio | weights |

> ⚠️ Untested on the real data (see README). Each step prints what it did; optional steps print a clear message if skipped.

In [ ]:
# Kaggle images already ship torch / transformers / librosa / lightgbm. Missing pieces (needs Internet ON):
!pip install -q faster-whisper sentence-transformers language-tool-python
# With Internet OFF: attach the wheels as a Kaggle Dataset and run:  !pip install --no-index --find-links=/kaggle/input/<wheels> faster-whisper

In [ ]:
# =====================  CONFIG  -- the ONLY cell you should need to edit  =====================
import sys
from pathlib import Path

CONFIG = dict(
    # ---- where the data is (Kaggle mounts it read-only; nothing is uploaded anywhere) ----
    DATA_DIR        = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final",
    TRAIN_CSV       = None,    # None -> auto-discover train.csv under DATA_DIR
    TEST_CSV        = None,
    SAMPLE_SUB_CSV  = None,
    TRAIN_AUDIO_DIR = None,    # None -> auto-discover (folder with 'train' in its name that holds audio)
    TEST_AUDIO_DIR  = None,
    ID_COL          = None,    # None -> auto-detect the file-name column
    LABEL_COL       = None,    # None -> auto-detect the numeric label column
    # ---- where caches / results go ----
    CACHE_DIR       = "/kaggle/working/cache",
    OUTPUT_DIR      = "/kaggle/working",
    # ---- models: hub name (Internet ON)  OR  local / Kaggle-Models path (Internet OFF, set OFFLINE=True) ----
    OFFLINE         = False,
    MODEL_CACHE_DIR = None,
    WHISPER_MODEL   = "small",      # faster-whisper size, or path to a CTranslate2 Whisper folder
    BASELINE_TEXT_ENCODER = "sentence-transformers/all-MiniLM-L6-v2",
    TEXT_ENCODER    = "sentence-transformers/all-mpnet-base-v2",
    HF_ENCODER      = "roberta-base",
    LM_NAME         = "gpt2",
    AUDIO_ENCODER   = "microsoft/wavlm-base-plus",
    LLM_NAME        = "Qwen/Qwen2.5-1.5B-Instruct",
    # ---- switches ----
    RUN_LANGUAGETOOL = True,        # auto-skipped if Java / download is unavailable
    RUN_LLM_RUBRIC   = False,       # optional zero-shot rubric feature
    FORCE_RECOMPUTE  = False,       # True -> ignore cached feature files
    # ---- reproducibility / validation ----
    SEED = 42, CV_SPLITS = 5, CV_REPEATS = 3,
)

def _find_repo():
    cands = [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/shl-grammar-scoring")]
    for pat in ("*", "*/*", "*/*/*", "*/*/*/*"):            # Kaggle nests datasets 1-4 levels deep
        cands += list(Path("/kaggle/input").glob(pat))
    for c in cands:
        if (c / "src" / "config.py").exists():
            return c
    raise FileNotFoundError("Cannot find the repo's src/ folder. Add the repo as a Kaggle Dataset (or clone it) and set REPO_DIR by hand.")

REPO_DIR = _find_repo()
sys.path.insert(0, str(REPO_DIR))
from src.config import Config
cfg = Config(**CONFIG).apply_env().seed_everything()      # env vars FIRST, before torch/transformers are imported
print("repo:", REPO_DIR)
print(cfg.summary())

## 1 · Load files, preprocess audio, inspect audio files

In [ ]:
import numpy as np, pandas as pd
from IPython.display import display
from src.data import load_competition
from src.audio import preprocess_all, scan_audio_info
from src.utils import keys_hash, free_gpu

data = load_competition(cfg)
meta = pd.concat([data["train"], data["test"]], ignore_index=True)
meta = preprocess_all(meta, cfg)

fd = cfg.features_dir
FORCE = cfg.FORCE_RECOMPUTE
kh = fd / "keys_hash.txt"
if kh.exists() and kh.read_text() != keys_hash(meta):
    print("File list changed since the last run -> all features will be recomputed.")
    FORCE = True
kh.write_text(keys_hash(meta))
def todo(fname):                         # True if this cached file must be (re)computed
    return FORCE or not (fd / fname).exists()

# label is stored for train rows only; test labels are never used (they are random placeholders)
meta[["split", "id", "stem", "label"]].to_csv(fd / "meta.csv", index=False)
if todo("audio_info.csv"):
    scan_audio_info(meta).to_csv(fd / "audio_info.csv", index=False)
print("rows:", len(meta), "| files that failed preprocessing:", int((~meta.pre_ok).sum()))

## 2 · Whisper transcripts + timestamps (cached; skipped if `00_baseline` already ran)

In [ ]:
from src.transcribe import transcribe_all
records = transcribe_all(meta, cfg)
free_gpu()
keys = list(zip(meta["split"], meta["stem"]))
recs = [records[k] for k in keys]
texts = [r["text"] for r in recs]
print("empty transcripts:", sum(t == "" for t in texts))
# (transcript text is deliberately NOT printed: it is derived from the competition data and must not be published)
print("words per transcript (min / median / max):", min(len(t.split()) for t in texts), "/",
      sorted(len(t.split()) for t in texts)[len(texts) // 2], "/", max(len(t.split()) for t in texts))

## 3 · Text statistics and fluency features (cheap, no models)
Everything below is a plain count or ratio, so each can be explained in one sentence.

In [ ]:
from src.text_features import text_feature_table
from src.fluency import fluency_table

hand = text_feature_table(texts)
flu = fluency_table(recs, cfg.PAUSE_THR, cfg.LONG_PAUSE_THR)
print("hand:", hand.shape, "| fluency:", flu.shape)
display(pd.concat([hand, flu], axis=1).describe().T.round(3).head(40))

## 4 · Language-model perplexity (GPT-2)
Average surprise of each token under GPT-2. Fluent, grammatical English tends to be less surprising.

In [ ]:
from src.lm_features import lm_features
if todo("lm.csv"):
    lm = lm_features(texts, cfg); lm.to_csv(fd / "lm.csv", index=False); free_gpu()
else:
    lm = pd.read_csv(fd / "lm.csv"); print("loaded cached lm.csv")
display(lm.describe().T.round(3))

## 5 · LanguageTool grammar errors — OPTIONAL
Needs Java and a first-run download. If anything fails, the group is simply skipped and the pipeline continues.

In [ ]:
from src.grammar_tool import languagetool_features
lt = None
if todo("lt.csv"):
    lt = languagetool_features(texts, cfg.RUN_LANGUAGETOOL)
    if lt is not None:
        lt.to_csv(fd / "lt.csv", index=False)
elif cfg.RUN_LANGUAGETOOL:
    lt = pd.read_csv(fd / "lt.csv"); print("loaded cached lt.csv")
print("LanguageTool features:", "SKIPPED" if lt is None else lt.shape)

## 6 · Transcript embeddings (sentence encoder + RoBERTa mean-pool)

In [ ]:
from src.embeddings import embed_sentence_transformer, embed_hf_meanpool

if todo("text_st.npy"):
    np.save(fd / "text_st.npy", embed_sentence_transformer(texts, cfg.TEXT_ENCODER, cfg)); free_gpu()
if todo("text_hf.npy"):
    np.save(fd / "text_hf.npy", embed_hf_meanpool(texts, cfg.HF_ENCODER, cfg)); free_gpu()
print({k: np.load(fd / f"{k}.npy").shape for k in ("text_st", "text_hf")})

## 7 · Audio embeddings (WavLM)
Mean and standard deviation over time of WavLM's last layer, from the **preprocessed** 16 kHz audio. This captures how the speech *sounds* (pronunciation, rhythm); it contains no explicit grammar information, so the ablation in `final.ipynb` will show whether it helps.

In [ ]:
from src.embeddings import embed_wavlm
if todo("audio_wavlm.npy"):
    np.save(fd / "audio_wavlm.npy", embed_wavlm(meta["cache_path"].tolist(), cfg)); free_gpu()
print("audio_wavlm:", np.load(fd / "audio_wavlm.npy").shape)

## 8 · Optional: local LLM scores each transcript against the rubric (zero-shot)
Off by default (`RUN_LLM_RUBRIC=False`). No labels are used, so it cannot leak; keep it only if the ablation in `final.ipynb` shows a CV gain.

In [ ]:
from src.llm_rubric import llm_rubric_features
llm = None
if cfg.RUN_LLM_RUBRIC:
    if todo("llm.csv"):
        llm = llm_rubric_features(texts, cfg)
        if llm is not None:
            llm.to_csv(fd / "llm.csv", index=False); free_gpu()
    else:
        llm = pd.read_csv(fd / "llm.csv"); print("loaded cached llm.csv")
print("LLM rubric feature:", "SKIPPED" if llm is None else llm.shape)

## 9 · Assemble and verify

In [ ]:
parts = [hand, flu, lm] + [p for p in (lt, llm) if p is not None]
features = pd.concat(parts, axis=1)
assert len(features) == len(meta) == len(np.load(fd / "text_st.npy")) == len(np.load(fd / "audio_wavlm.npy"))
features.to_csv(fd / "features.csv", index=False)

groups = {}
for c in features.columns:
    groups.setdefault(c.split("_")[0], []).append(c)
print("feature groups:", {g: len(c) for g, c in groups.items()})
print("columns with missing values:\n", features.isna().sum()[features.isna().sum() > 0].to_string())
print("\nsaved to", fd)
print(sorted(p.name for p in fd.iterdir()))

Done. Now run `final.ipynb`. If it runs in a **new** Kaggle session, save this notebook's output as a Dataset (keep it private — it is derived from the competition data), attach it, and point `CONFIG["CACHE_DIR"]` at it.